In [1]:
import pandas as pd
import numpy as np
import os
import sys
%load_ext autoreload
%autoreload 2
# sys.path.append('/home/wolfgang/repos/sleep_research_io')
# from sleep_research_functions import *
%matplotlib widget
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu

from icu_sleep_breathing_2020_help_functions import * 

pd.options.display.max_rows = 300
pd.options.display.max_columns = 300

fontsize = 8
font = {'family' : 'normal',
        'weight' : 'normal',
        'size'   : fontsize}
import matplotlib
matplotlib.rc('font', **font)

### load summary tables:

In [2]:
plots_savedir = '/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/plots'

In [3]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(sleeplab_matched=True)

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
# of subjects ICU after inclusion criteria: 103
# of 12-hour segments ICU after inclusion criteria: 621
24-hour segments: 256
12-hour segments: 365

# of subjects sleeplab before inclusion criteria: 307
# of 12-hour segments sleeplab before inclusion criteria: 307
# of subjects sleeplab after inclusion criteria: 307
# of 12-hour segments sleeplab after before inclusion criteria: 307


In [4]:
[x for x in summary_subjects_icu if 'airgo' in x]
airgo_cols = [f'f{i}_airgo_available_h' for i in range(1,10)]
airgo_avail = summary_subjects_icu[airgo_cols].copy()
airgo_avail[pd.isna(airgo_avail)] = 0
airgo_avail = (airgo_avail>0).astype(int)
airgo_avail = pd.DataFrame(airgo_avail.sum(axis=1).values, columns=['airgo_days'])

In [5]:
airgo_avail['study_id'] = summary_subjects_icu['study_id'].values
airgo_avail['first_day'] = summary_subjects_icu['f2_timerange'].values

In [8]:
airgo_avail[['study_id', 'airgo_days', 'first_day']].to_csv('icu_sleep_airgo_days.csv', index=False)

In [14]:
matched_columns = [x for x in summary_days_sleeplab.columns if 'match' in x]

demographics_icu = summary_subjects_icu[['study_id', 'mrn', 'age', 'sex']].copy()
demographics_icu['population'] = 'ICU'
for col in matched_columns:
    demographics_icu[col] = 1 # set matched to true for all icu patients.
demographics_sleeplab = summary_days_sleeplab[['study_id', 'mrn', 'age', 'sex'] + matched_columns].copy()
demographics_sleeplab['sex'] = (demographics_sleeplab['sex'] == 'Male').astype(int)
demographics_sleeplab['population'] = 'Sleeplab'

In [15]:
demographics_all = pd.concat([demographics_sleeplab, demographics_icu], axis=0, sort=False)
print(demographics.head(2))
print(demographics.tail(2))

    study_id        mrn     age  sex  matched_all  matched_ahi_0_5  \
17        28  2750570.0  58.529  1.0            1                0   
20        31  4997145.0  67.781  0.0            1                0   

    matched_ahi_5_15  matched_ahi_15_30  matched_ahi_30_100  \
17                 0                  0                   1   
20                 0                  0                   1   

    matched_ahi_15_100 population  
17                   1   Sleeplab  
20                   1   Sleeplab  
     study_id        mrn   age  sex  matched_all  matched_ahi_0_5  \
106       188  5407879.0  62.0  0.0            1                1   
107       189  3243449.0  76.0  0.0            1                1   

     matched_ahi_5_15  matched_ahi_15_30  matched_ahi_30_100  \
106                 1                  1                   1   
107                 1                  1                   1   

     matched_ahi_15_100 population  
106                   1        ICU  
107             

## ICU vs Sleeplab: Matching and Age & Sex distribution  
similar as in Matching_ICUSleep_Sleeplab.ipynb

In [16]:
ahi_categories = ['all', 'ahi_0_5', 'ahi_5_15', 'ahi_15_30', 'ahi_30_100']
ahi_names = {
    'ahi_0_5': 'AHI < 5',
    'ahi_5_15': '5 < AHI <=15',
    'ahi_15_30': '15 < AHI <= 30',
    'ahi_30_100': 'AHI > 30',
}

for ahi_category in ahi_categories:
    
    print(f'SLEEPLAB AHI SELECTION: {ahi_category}')
    # AHI selection
    demographics = demographics_all.loc[demographics_all['matched_' + ahi_category] == 1, :]
    
    ylim = [44, 105]
    ylabel = 'Age (years)'
    xlabel = ''
    x_data = 'population'
    y_data = 'age'
    color_boxplot = 'white'
    color_swarme = np.array([255,185,0])/255 # 'gold'
    color_swarmf= np.array([255,185,0])/255 # 'gold'
    swarmsize = 1
    swarmalpha = 0.8 
    
    # PLOTS & hypothesis tests:
    plt.figure(figsize=(7,3))
    plt.subplot(131)
    plt.title('All Subjects', size = fontsize)
    ax = sns.boxplot(x=x_data, y=y_data, data=demographics, width=0.85, fliersize=0, color=color_boxplot, saturation=1)
    ax = sns.swarmplot(x=x_data, y=y_data, data=demographics, color=color_swarmf, edgecolor=color_swarme, linewidth=1, size=swarmsize, alpha = swarmalpha)
    if ahi_category == 'all':
        ax.set_xticklabels([f"Sleeplab \n(n={np.sum(demographics.population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics.population=='ICU')})"])
    else:
        ax.set_xticklabels([f"Sleeplab \n{ahi_names[ahi_category]}\n(n={np.sum(demographics.population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics.population=='ICU')})"])

    ax.set_ylim(ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)

    plt.subplot(132)
    plt.title('Female Subjects', size = fontsize)
    ax = sns.boxplot(x=x_data, y=y_data, data=demographics[demographics.sex==0], width=0.85, fliersize=0, color=color_boxplot)
    ax = sns.swarmplot(x=x_data, y=y_data, data=demographics[demographics.sex==0], color=color_swarmf, edgecolor=color_swarme, linewidth=1, size=swarmsize, alpha = swarmalpha)
    if ahi_category == 'all':
        ax.set_xticklabels([f"Sleeplab \n(n={np.sum(demographics[demographics.sex==0].population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics[demographics.sex==0].population=='ICU')})"])
    else:
        ax.set_xticklabels([f"Sleeplab \n{ahi_names[ahi_category]}\n(n={np.sum(demographics[demographics.sex==0].population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics[demographics.sex==0].population=='ICU')})"])

    ax.set_ylim(ylim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    plt.subplot(133)
    plt.title('Male Subjects', size = fontsize)
    ax = sns.boxplot(x=x_data, y=y_data, data=demographics[demographics.sex==1], width=0.85, fliersize=0, color=color_boxplot)
    ax = sns.swarmplot(x=x_data, y=y_data, data=demographics[demographics.sex==1], color=color_swarmf, edgecolor=color_swarme, linewidth=1, size=swarmsize,
                      alpha = swarmalpha)
    if ahi_category == 'all':
        ax.set_xticklabels([f"Sleeplab \n(n={np.sum(demographics[demographics.sex==1].population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics[demographics.sex==1].population=='ICU')})"])
    else:
        ax.set_xticklabels([f"Sleeplab \n{ahi_names[ahi_category]}\n(n={np.sum(demographics[demographics.sex==1].population=='Sleeplab')})", f"ICU \n(n={np.sum(demographics[demographics.sex==1].population=='ICU')})"])

    ax.set_ylim(ylim)
    plt.tight_layout()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    
    sel = demographics.loc[(demographics.sex==0) & (demographics.population=='ICU'), 'age']
    n_female_icu = sel.shape[0]
    sel = demographics.loc[(demographics.sex==0) & (demographics.population=='Sleeplab'), 'age']
    n_female_sleeplab = sel.shape[0]

    sel = demographics.loc[(demographics.sex==1) & (demographics.population=='ICU'), 'age']
    n_male_icu = sel.shape[0]
    sel = demographics.loc[(demographics.sex==1) & (demographics.population=='Sleeplab'), 'age']
    n_male_sleeplab = sel.shape[0]
    print('')
    print(f'n_female_icu:      {n_female_icu}')
    print(f'n_female_sleeplab: {n_female_sleeplab}')
    print(f'n_male_icu:        {n_male_icu}')
    print(f'n_male_sleeplab:   {n_male_sleeplab}')
    print('')
    test_samples = []
    sel = demographics.loc[demographics.population=='ICU', 'age']
    test_samples.append(sel.values)
    print(f"Median age for ICU:             {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    sel = demographics.loc[demographics.population=='Sleeplab', 'age']
    test_samples.append(sel.values)
    print(f"Median age for sleeplab:        {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
    print(f't-test (Welch): p-value:        {p_value:.2f}')
    t_statistic, p_value = mannwhitneyu(test_samples[0], test_samples[1])
    print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')

    print('')
    test_samples = []
    sel = demographics.loc[(demographics.sex == 0 ) & (demographics.population=='ICU'), 'age']
    test_samples.append(sel.values)
    print(f"Median age for ICU female:      {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    sel = demographics.loc[(demographics.sex == 0) & (demographics.population=='Sleeplab'), 'age']
    test_samples.append(sel.values)
    print(f"Median age for sleeplab female: {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
    print(f't-test (Welch): p-value:        {p_value:.2f}')
    t_statistic, p_value = mannwhitneyu(test_samples[0], test_samples[1])
    print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')
    test_samples = []
    sel = demographics.loc[(demographics.sex == 1) & (demographics.population=='ICU'), 'age']
    test_samples.append(sel.values)
    print(f"\nMedian age for ICU male:        {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    sel = demographics.loc[(demographics.sex == 1) & (demographics.population=='Sleeplab'), 'age']
    test_samples.append(sel.values)
    print(f"Median age for sleeplab male:   {sel.median():.1f} , IQR: [{sel.quantile(0.25):.1f}, {sel.quantile(0.75):.1f}]")
    t_statistic, p_value = ttest_ind(test_samples[0], test_samples[1], equal_var=False)
    print(f't-test (Welch): p-value:        {p_value:.2f}')
    t_statistic, p_value = mannwhitneyu(test_samples[0], test_samples[1])
    print(f'Mann Whiteney U test: p-value:  {p_value:.2f}')

    print(f"\nMale/Female Ratio ICU:          {n_male_icu / n_female_icu:.2f}")
    print(f"Male/Female Ratio Sleeplab:     {n_male_sleeplab / n_female_sleeplab:.2f}")

    print('')

    plt.savefig(f'Figure_Age_Sex_Distribution_ahi_{ahi_category}.jpg', dpi=500)
    plt.savefig(f'Figure_Age_Sex_Distribution_ahi_{ahi_category}.svg')


SLEEPLAB AHI SELECTION: all


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":



n_female_icu:      41
n_female_sleeplab: 98
n_male_icu:        62
n_male_sleeplab:   122

Median age for ICU:             68.0 , IQR: [62.5, 75.0]
Median age for sleeplab:        67.3 , IQR: [60.6, 73.4]
t-test (Welch): p-value:        0.78
Mann Whiteney U test: p-value:  0.32

Median age for ICU female:      67.0 , IQR: [57.0, 76.0]
Median age for sleeplab female: 66.2 , IQR: [59.9, 71.9]
t-test (Welch): p-value:        0.88
Mann Whiteney U test: p-value:  0.42

Median age for ICU male:        68.0 , IQR: [64.0, 75.0]
Median age for sleeplab male:   68.3 , IQR: [61.9, 74.2]
t-test (Welch): p-value:        0.92
Mann Whiteney U test: p-value:  0.43

Male/Female Ratio ICU:          1.51
Male/Female Ratio Sleeplab:     1.24

SLEEPLAB AHI SELECTION: ahi_0_5


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

...

/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":



n_female_icu:      41
n_female_sleeplab: 13
n_male_icu:        62
n_male_sleeplab:   26

Median age for ICU:             68.0 , IQR: [62.5, 75.0]
Median age for sleeplab:        71.5 , IQR: [60.4, 74.9]
t-test (Welch): p-value:        0.62
Mann Whiteney U test: p-value:  0.27

Median age for ICU female:      67.0 , IQR: [57.0, 76.0]
Median age for sleeplab female: 70.9 , IQR: [60.2, 75.0]
t-test (Welch): p-value:        0.57
Mann Whiteney U test: p-value:  0.31

Median age for ICU male:        68.0 , IQR: [64.0, 75.0]
Median age for sleeplab male:   71.9 , IQR: [61.0, 74.7]
t-test (Welch): p-value:        0.98
Mann Whiteney U test: p-value:  0.40

Male/Female Ratio ICU:          1.51
Male/Female Ratio Sleeplab:     2.00

SLEEPLAB AHI SELECTION: ahi_30_100


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:3000: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  if edgecolor == "gray":



n_female_icu:      41
n_female_sleeplab: 5
n_male_icu:        62
n_male_sleeplab:   13

Median age for ICU:             68.0 , IQR: [62.5, 75.0]
Median age for sleeplab:        64.5 , IQR: [53.2, 71.8]
t-test (Welch): p-value:        0.30
Mann Whiteney U test: p-value:  0.06

Median age for ICU female:      67.0 , IQR: [57.0, 76.0]
Median age for sleeplab female: 73.8 , IQR: [70.0, 77.3]
t-test (Welch): p-value:        0.14
Mann Whiteney U test: p-value:  0.04

Median age for ICU male:        68.0 , IQR: [64.0, 75.0]
Median age for sleeplab male:   55.2 , IQR: [52.3, 66.8]
t-test (Welch): p-value:        0.00
Mann Whiteney U test: p-value:  0.00

Male/Female Ratio ICU:          1.51
Male/Female Ratio Sleeplab:     2.60



### Table 1

In [17]:
[summary_subjects_icu, summary_subjects_sleeplab, 
 summary_days_icu, summary_days_sleeplab] = load_summary_data_with_inclusion_criteria(sleeplab_matched=True)

# of subjects ICU before inclusion criteria: 108
# of 12-hour segments ICU before inclusion criteria: 1230
# of subjects ICU after inclusion criteria: 103
# of 12-hour segments ICU after inclusion criteria: 621
24-hour segments: 256
12-hour segments: 365

# of subjects sleeplab before inclusion criteria: 307
# of 12-hour segments sleeplab before inclusion criteria: 307
# of subjects sleeplab after inclusion criteria: 307
# of 12-hour segments sleeplab after before inclusion criteria: 307


In [18]:
summary_subjects_icu

print('Table 1.')
print('\nICU:')
print(f'N:                     {summary_subjects_icu.shape[0]}')
print(f'Age Mean +- STD:        {summary_subjects_icu.age.mean():.1f}  +- {summary_subjects_icu.age.std():.1f}')
print(f'# Male | # Female:      {sum(summary_subjects_icu.sex == 1)}  |  {sum(summary_subjects_icu.sex == 0)}')
print(f'% Male:                 {sum(summary_subjects_icu.sex == 1)/summary_subjects_icu.shape[0]:.2f}')
print(f'BMI Mean +- STD:        {summary_subjects_icu.bmi.mean():.1f}  +- {summary_subjects_icu.bmi.std():.1f}')
print(f'ICU LOS Mean +- STD:    {summary_subjects_icu.icu_los.mean():.1f}  +- {summary_subjects_icu.icu_los.std():.1f}')
print(f'HOS LOS Mean +- STD:    {summary_subjects_icu.hospital_los.mean():.1f}  +- {summary_subjects_icu.hospital_los.std():.1f}')
print(f'CCI Mean +- STD:        {summary_subjects_icu.CCI.mean():.1f}  +- {summary_subjects_icu.CCI.std():.1f}')

print(summary_days_sleeplab.shape)
print('Unique patients selection here.')
summary_days_sleeplab = summary_days_sleeplab.drop_duplicates(subset=['mrn'])
summary_days_sleeplab['sex'] = summary_days_sleeplab['sex'] == 'Male'
print('\nSleeplab:')
print(f'N:                     {summary_days_sleeplab.shape[0]}')
print(f'Age Mean +- STD:        {summary_days_sleeplab.age.mean():.1f}  +- {summary_days_sleeplab.age.std():.1f}')
print(f'# Male | # Female:      {sum(summary_days_sleeplab.sex == 1)}  |  {sum(summary_days_sleeplab.sex == 0)}')
print(f'% Male:                 {sum(summary_days_sleeplab.sex == 1)/summary_days_sleeplab.shape[0]:.2f}')
print(f'CCI Mean +- STD:        {summary_days_sleeplab.cci.mean():.1f}  +- {summary_days_sleeplab.cci.std():.1f}')
print(f'AHI Mean +- STD:        {summary_days_sleeplab.ahi_expert.mean():.1f}  +- {summary_days_sleeplab.ahi_expert.std():.1f}')


Table 1.

ICU:
N:                     103
Age Mean +- STD:        68.0  +- 8.9
# Male | # Female:      62  |  41
% Male:                 0.60
BMI Mean +- STD:        27.3  +- 6.1
ICU LOS Mean +- STD:    4.9  +- 4.2
HOS LOS Mean +- STD:    14.3  +- 12.6
CCI Mean +- STD:        2.3  +- 2.2
(307, 192)
Unique patients selection here.

Sleeplab:
N:                     300
Age Mean +- STD:        63.8  +- 10.7
# Male | # Female:      173  |  127
% Male:                 0.58
CCI Mean +- STD:        1.6  +- 1.7
AHI Mean +- STD:        9.7  +- 10.4


In [124]:
summary_days_sleeplab['sex'].unique()

array(['Male', 'Female', 'Unknown'], dtype=object)

## Signal Availabilities

In [19]:
summary_days_sleeplab = summary_days_sleeplab.loc[summary_days_sleeplab.matched_all == 1, :]
print(summary_days_sleeplab.shape)

(214, 192)


In [20]:
summary_f_icu = summary_days_icu.loc[summary_days_icu.day_cat == 'f', :]
summary_dn_icu = summary_days_icu.loc[summary_days_icu.day_cat != 'f', :]

In [21]:
fig, ax = plt.subplots(1,4,figsize=(8,4))
for ax_tmp in ax:
    ax_tmp.spines['top'].set_visible(False)
    ax_tmp.spines['right'].set_visible(False)
    ax_tmp.spines['bottom'].set_visible(False)
    ax_tmp.spines['left'].set_visible(False)
    ax_tmp.set_yticks([])

sns.boxplot(y='airgo_available_h', data=summary_dn_icu, width=0.9, fliersize=0, ax=ax[0], color='blue')
sns.swarmplot(y='airgo_available_h', data=summary_dn_icu, color='gray', edgecolor='black', linewidth=1, ax=ax[0], size=2, alpha=0.7)

sns.boxplot(y='ecg_available', data=summary_dn_icu, width=0.9, fliersize=0, ax=ax[1], color='red')
sns.swarmplot(y='ecg_available', data=summary_dn_icu, color='gray', edgecolor='black', linewidth=1, ax=ax[1], size=2, alpha=0.7)

sns.boxplot(y='airgo_available_h', data=summary_days_sleeplab, width=0.9, fliersize=0, ax=ax[2], color='blue')
sns.swarmplot(y='airgo_available_h', data=summary_days_sleeplab, color='gray', edgecolor='black', linewidth=1, ax=ax[2], size=2, alpha=0.7)

sns.boxplot(y='ecg_available', data=summary_days_sleeplab, width=0.9, fliersize=0, ax=ax[3], color='red')
sns.swarmplot(y='ecg_available', data=summary_days_sleeplab, color='gray', edgecolor='black', linewidth=1, ax=ax[3], size=2, alpha=0.7)

ax[0].set_xticklabels([f'ICU (n={summary_dn_icu.shape[0]})\n Respiration available'])
ax[0].set_ylim([0,12])
ax[0].set_yticks([0, 4, 8, 12])

ax[1].set_xticklabels([f'ICU (n={summary_dn_icu.shape[0]}) \n ECG available'])
ax[2].set_xticklabels([f'Sleeplab (n={summary_days_sleeplab.shape[0]})\n Respiration available'])
ax[3].set_xticklabels([f'Sleeplab (n={summary_days_sleeplab.shape[0]})\n ECG available'])

for ax_tmp in ax:
    ax_tmp.set_ylabel('')
ax[0].set_ylabel('Duration (h)')

plt.savefig(os.path.join(plots_savedir, 'Data_Availability_Overview.jpg'), dpi=500)
plt.savefig(os.path.join(plots_savedir, 'Data_Availability_Overview.svg'))

print('Data Availability (Hours) Overview ECG and Respiration Signals:')
print(f"ICU Respiration       Mean (+- STD)  {summary_dn_icu.airgo_available_h.mean():.2f} (+-{summary_dn_icu.airgo_available_h.std():.2f})")
print(f"ICU ECG               Mean (+- STD)  {summary_dn_icu.ecg_available.mean():.2f} (+-{summary_dn_icu.ecg_available.std():.2f})")
print(f"Sleeplab Respiration  Mean (+- STD)  {summary_days_sleeplab.airgo_available_h.mean():.2f} (+-{summary_days_sleeplab.airgo_available_h.std():.2f})")
print(f"Sleeplab ECG          Mean (+- STD)  {summary_days_sleeplab.ecg_available.mean():.2f} (+-{summary_days_sleeplab.ecg_available.std():.2f})")


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:1296: UserWarning: 48.2% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)
/home/wolfgang/anaconda3/envs/analysis/lib/python3.6/site-packages/seaborn/categorical.py:1296: UserWarning: 45.2% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.
  warnings.warn(msg, UserWarning)


Data Availability (Hours) Overview ECG and Respiration Signals:
ICU Respiration       Mean (+- STD)  9.88 (+-3.23)
ICU ECG               Mean (+- STD)  10.95 (+-1.88)
Sleeplab Respiration  Mean (+- STD)  7.34 (+-0.97)
Sleeplab ECG          Mean (+- STD)  7.45 (+-0.75)
